# RAG för klassificering av SNI 2025

pip install torch sentence-transformers transformers openpyxl

In [1]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df_sni2025 = pd.read_excel("https://minio.lab.sspcloud.fr/scbmrid/OmfattartexterSNI2025.xlsx")

df_sni2025.head()

,Niva,SNI2025,Rubrik,Omfattar,Omfattar_aven,Omfattar_inte,Special
0,1,A,"JORDBRUK, SKOGSBRUK OCH FISKE",I denna avdelning ingår exploatering av naturt...,"ekologiskt jordbruk, vattenbruk, odling av gen...",diverse varor som hushållen producerar för hus...,NaN
1,2,1,Jordbruk och jakt samt stödverksamhet i anslut...,"två grundläggande aktiviteter, produktion av v...",stödverksamhet av underordnad betydelse i jord...,vidarebearbetning av jordbruksprodukter (redov...,NaN
2,4,11,Odling av ett- och tvååriga växter,"odling av ett- och tvååriga växter, dvs. växte...",NaN,NaN,NaN
3,5,111,"Odling av spannmål (utom ris), baljväxter och ...","alla former av odling av spannmål, baljväxter ...",NaN,"odling av ris, jfr 01.12 - odling av sockermaj...",NaN
4,6,1110,"Odling av spannmål (utom ris), baljväxter och ...","alla former av odling av spannmål, baljväxter ...",NaN,"odling av ris, jfr 01.120 - odling av sockerma...",NaN


In [17]:
df_sni2025_two = df_sni2025[df_sni2025["Niva"] == 2].reset_index(drop=True)

df_sni2025_two.head()

,Niva,SNI2025,Rubrik,Omfattar,Omfattar_aven,Omfattar_inte,Special
0,2,1,Jordbruk och jakt samt stödverksamhet i anslut...,"två grundläggande aktiviteter, produktion av v...",stödverksamhet av underordnad betydelse i jord...,vidarebearbetning av jordbruksprodukter (redov...,NaN
1,2,2,Skogsbruk,skogsodling och produktion av rundvirke såväl ...,NaN,vidare bearbetning av trä som börjar med sågni...,NaN
2,2,3,Fiske och vattenbruk,fiske och vattenbruk i avsikt att fånga eller ...,stödverksamhet av underordnad betydelse för fi...,"bearbetning av fisk, kräftdjur eller blötdjur,...",NaN
3,2,5,Kolutvinning,utvinning av fasta mineralbränslen genom gruvb...,NaN,"framställning av koks, jfr 19.10 - tjänster i ...",NaN
4,2,6,Utvinning av råpetroleum och naturgas,"produktion av råolja, brytning och utvinning a...",NaN,"stödverksamhet för olje- och gasfält, utförd m...",NaN


In [18]:
codes = df_sni2025_two["SNI2025"].tolist()
titles = df_sni2025_two["Rubrik"].tolist()
descriptions = df_sni2025_two["Omfattar"].tolist()

descriptor_template = "{title}.\n{description}"

descriptors = []
for title, description in zip(titles, descriptions):
    descriptors.append(
        descriptor_template.format(title=title.upper(), description=description)
    )

In [19]:
print(descriptors[5])

UTVINNING AV METALLMALMER.
utvinning av metallmineral (malmer) genom gruv- eller markbrytning, utvinning på havs- eller sjöbotten o.d.


In [20]:
model = SentenceTransformer("KBLab/sentence-bert-swedish-cased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4975.72it/s]
BertModel LOAD REPORT from: KBLab/sentence-bert-swedish-cased
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
embeddings = model.encode(descriptors, convert_to_tensor=True)

In [22]:
def search_base(
    query: str,
    base: torch.Tensor,
    embedding_model: SentenceTransformer,
    top_k: int = 5
):
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(base, query_embedding)
    top_results = torch.topk(cos_scores, k=top_k)

    return top_results

In [23]:
sample_query = "Verksamhet som avser utvinning av mineraler"


top_results = search_base(sample_query, embeddings, model, top_k=5)

for i, sim in zip(top_results.indices, top_results.values):
    code = df_sni2025_two.iloc[int(i)]["SNI2025"]
    title = df_sni2025_two.iloc[int(i)]["Rubrik"]
    print(f"{code}: {title}\nSimilarity: {sim:.3f}\n")

7: Utvinning av metallmalmer
Similarity: 0.850

5: Kolutvinning
Similarity: 0.790

8: Annan utvinning av mineral
Similarity: 0.778

9: Stödverksamhet avseende utvinning
Similarity: 0.732

6: Utvinning av råpetroleum och naturgas
Similarity: 0.680

